In [6]:
import pandas as pd
import numpy as np
import re
import string
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


df = pd.read_csv("IMDB Dataset.csv", engine='python', on_bad_lines='skip')

print("Shape:", df.shape)
print(df.head())

print("\nClass Distribution:")
print(df['sentiment'].value_counts())



stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):

    text = text.lower()


    text = re.sub(r'http\S+', '', text)


    text = text.translate(str.maketrans('', '', string.punctuation))


    tokens = word_tokenize(text)


    tokens = [word for word in tokens if word not in stop_words]

    tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(tokens)

df['clean_text'] = df['review'].apply(preprocess)

print(df[['review', 'clean_text']].head())


X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'], test_size=0.2, random_state=42
)


bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)



models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier()
}


def evaluate(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted')
    rec = recall_score(y_test, preds, average='weighted')
    f1 = f1_score(y_test, preds, average='weighted')

    return acc, prec, rec, f1


results = []

print("\n--- BoW Results ---")
for name, model in models.items():
    acc, prec, rec, f1 = evaluate(model, X_train_bow, X_test_bow, y_train, y_test)
    results.append([name, "BoW", acc, prec, rec, f1])
    print(f"{name}: Accuracy={acc:.4f}")

print("\n--- TF-IDF Results ---")
for name, model in models.items():
    acc, prec, rec, f1 = evaluate(model, X_train_tfidf, X_test_tfidf, y_train, y_test)
    results.append([name, "TF-IDF", acc, prec, rec, f1])
    print(f"{name}: Accuracy={acc:.4f}")



results_df = pd.DataFrame(results, columns=[
    "Model", "Vectorizer", "Accuracy", "Precision", "Recall", "F1 Score"
])

print("\nFinal Comparison:")
print(results_df)


best_model = results_df.sort_values(by="F1 Score", ascending=False).iloc[0]

print("\nBest Model:")
print(best_model)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Shape: (24514, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Class Distribution:
sentiment
negative    12291
positive    12223
Name: count, dtype: int64
                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                          clean_text  
0  one review mention watch 1 oz episod youll hoo...  
1  wonder littl product br br film techniqu unass...  
2  though

## Conclusion

- TF-IDF performed better than BoW due to better word importance handling.
- Logistic Regression gave best results among models.
- Preprocessing improved accuracy significantly.
- Decision Trees overfitted in some cases.
- Naive Bayes was fast but slightly less accurate.

Final Best Combination:
TF-IDF + Logistic Regression